# Business Entity Resolution — A100 Implementation Notebook (Google Colab)

**Runs on:** Colab A100 runtime (40 GB VRAM, ~12 vCPU, high-RAM). This is the *training* notebook: bi-encoder fine-tuning, cross-encoder rerank, and large-scale blocking all run here on GPU.
**Siblings:** `entity_resolution.ipynb` (SageMaker t3.medium, sample-only) and `entity_resolution_local.ipynb` (Windows + Arc 140T + OpenVINO inference) — untouched. Pipeline logic (scorer, split, normalization, skeleton) is intentionally identical across all three; only budgets and the GPU stages differ.
**Plan ref:** `../ENTITY_RESOLUTION_PLAN.md` — Phase 0 → 0.5 → 1 → 2 → 3 (full, not stub) → 4/5. All `.tsv` reads use `sep="\t"`.

Colab survival rules: mount Drive first (cell below) — dataset, checkpoints, and both TSV outputs live there so an idle-timeout/recycle loses nothing. Expected data layout on Drive:
```
MyDrive/amlc/dataset/train/train_source1.tsv ...
```

In [ ]:
!pip install -q faiss-cpu rapidfuzz lightgbm sentence-transformers accelerate openvino optimum-intel

In [ ]:
import hashlib
import os
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch

IN_COLAB = os.path.exists("/content")
GPU = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"
VRAM = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0.0
print(f"colab={IN_COLAB} gpu={GPU} vram={VRAM:.1f} GB torch={torch.__version__}")
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > A100 GPU."

# ---- VRAM-based budgets (A100-40GB; smaller GPUs fall back) ----
if VRAM >= 35:
    SAMPLE_S1, POOL_DOCS, ENC_BATCH, FT_EPOCHS = 100000, 200000, 256, 2
    MODE = "a100-40GB"
else:
    SAMPLE_S1, POOL_DOCS, ENC_BATCH, FT_EPOCHS = 20000, 100000, 64, 1
    MODE = f"gpu-fallback-{GPU}"
TOP_K = 10
RANDOM_STATE = 42
RUN_FULL = False  # full 2.2M x 10M blocking: flip with Drive checkpointing on (see scale-up cell)
print(f"MODE={MODE} SAMPLE_S1={SAMPLE_S1} POOL_DOCS={POOL_DOCS} BATCH={ENC_BATCH}")

In [ ]:
# ---- Drive mount (persistence across recycles) + data root autodetect ----
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("drive mounted")
except Exception as e:
    print("drive mount skipped:", type(e).__name__)

BASE = Path("/content/drive/MyDrive/amlc") if Path("/content/drive").exists() else Path("/content")
DATA_ROOT, OUT_DIR, CKPT_DIR, OV_DIR = None, BASE / "output-a100", BASE / "checkpoints", BASE / "models/minilm-ov"
for c in [BASE / "dataset", Path("/content") / "dataset",
          Path("/content") / "student_resource" / "student_resource" / "dataset"]:
    if (c / "train" / "train_source1.tsv").exists():
        DATA_ROOT = c
        break
print("BASE =", BASE, "\nDATA_ROOT =", DATA_ROOT)
OUT_DIR.mkdir(parents=True, exist_ok=True); CKPT_DIR.mkdir(parents=True, exist_ok=True)
assert DATA_ROOT is not None, "Copy dataset/ to BASE and re-run."

## Phase 0 — Macro-F0.5 scorer (with unit test)
Singleton semantics: empty/empty → 1.0; any false merge on a singleton → 0.0.

In [ ]:
def f05_single(y_true: set, y_pred: set) -> float:
    tp = len(y_true & y_pred)
    prec = tp / len(y_pred) if y_pred else (1.0 if not y_true else 0.0)
    rec = tp / len(y_true) if y_true else (1.0 if not y_pred else 0.0)
    if prec + rec == 0:
        return 0.0
    return 1.25 * prec * rec / (0.25 * prec + rec)


def macro_f05(truth: dict, pred: dict) -> float:
    return sum(f05_single(set(truth[k]), set(pred.get(k, ()))) for k in truth) / len(truth)


# unit tests: PDF worked example S1-00001 -> P=2/3, R=1.0, F0.5=0.7142857
assert abs(f05_single({"S2-00047", "S3-00812"}, {"S2-00047", "S2-00193", "S3-00812"}) - 0.7142857) < 1e-6
assert f05_single(set(), set()) == 1.0
assert f05_single(set(), {"S2-1"}) == 0.0
assert f05_single({"S2-1"}, set()) == 0.0
print("scorer OK: pdf-example=0.7142857, singleton-empty=1.0, singleton-fp=0.0")

## Phase 0 — Validation split (hash-based, single chunked pass)
Deterministic 10% of S1 ids (`md5 % 10 == 0`). GT streams in chunks; only val rows kept (~220k).

In [ ]:
def in_val(s1: str) -> bool:
    return hashlib.md5(s1.encode()).digest()[0] % 10 == 0


def bucket(n: int) -> str:
    if n == 0:
        return "0-singleton"
    if n == 1:
        return "1"
    if n <= 3:
        return "2-3"
    if n <= 5:
        return "4-5"
    return "6+"


val_matches: dict = {}
total = 0
t0 = time.time()
for ch in pd.read_csv(DATA_ROOT / "train" / "train_ground_truth.tsv", sep="\t", chunksize=200000):
    s1s = ch["source1_entity_id"].astype(str)
    ms = ch["matched_entity_ids"].fillna("").astype(str)
    for s1, m in zip(s1s, ms):
        total += 1
        if in_val(s1):
            m = m.strip()
            val_matches[s1] = [x for x in m.split(",") if x] if m else []
print(f"GT rows={total} val_S1={len(val_matches)} in {time.time()-t0:.0f}s")
dist: dict = {}
for v in val_matches.values():
    b = bucket(len(v))
    dist[b] = dist.get(b, 0) + 1
print("val bucket distribution:", dist)

## Phase 0.5 — Walking skeleton (trivial PIN blocking, full pipeline)
Integration first: prove load → normalize → block → features → score → TSVs → validation PASS before the GPU stages.

In [ ]:
US_IN_ABBR = {"corp": "corporation", "inc": "incorporated", "pvt": "private",
               "ltd": "limited", "rd": "road", "st": "street", "ave": "avenue",
               "blvd": "boulevard", "ste": "suite", "apt": "apartment",
               "mfg": "manufacturing", "ent": "enterprises", "co": "company"}
FR_ABBR = {"sarl": "societe responsabilite limitee", "sas": "societe actions simplifiee",
           "sa": "societe anonyme", "eurl": "entreprise unipersonnelle",
           "rue": "rue", "bd": "boulevard", "av": "avenue", "pl": "place",
           "imp": "impasse", "cedex": "cedex", "ste": "societe", "ets": "etablissements"}
ABBR = {**US_IN_ABBR, **FR_ABBR}  # open-set: FR included despite 0% train coverage
PIN_RE = r"(?<!\d)(\d{5,6})(?!\d)"  # IN 6-digit, US/FR 5-digit


def normalize_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    s = s.lower().replace("&", " and ")
    s = re.sub(r"[^a-z0-9 ]", " ", s)
    toks = [ABBR.get(t, t) for t in s.split()]
    return re.sub(r"\s+", " ", " ".join(toks)).strip()


def extract_pin(addr: str) -> str:
    m = re.search(PIN_RE, str(addr))
    return m.group(1) if m else ""


assert normalize_text("SARL Dupont") == "societe responsabilite limitee dupont"
assert extract_pin("12 Rue de la Paix, 75002 Paris") == "75002"
assert normalize_text("Lumay Bóral") == normalize_text("Lumay Boral")
print("normalization OK (US/IN + FR + accents)")

In [ ]:
from rapidfuzz import fuzz

sample_s1 = sorted(val_matches)[:SAMPLE_S1]
sample_set = set(sample_s1)
print(f"skeleton queries: {len(sample_s1)}")

# 1. load sample S1 rows (chunked scan, keep targets only)
s1_parts = []
for ch in pd.read_csv(DATA_ROOT / "train" / "train_source1.tsv", sep="\t", chunksize=200000):
    hit = ch[ch["entity_id"].astype(str).isin(sample_set)]
    if len(hit):
        s1_parts.append(hit)
s1_df = pd.concat(s1_parts, ignore_index=True)
assert len(s1_df) == len(sample_s1), (len(s1_df), len(sample_s1))
s1_df["norm_name"] = s1_df["business_name"].fillna("").map(normalize_text)
s1_df["norm_addr"] = s1_df["business_address"].fillna("").map(normalize_text)
s1_df["pin"] = s1_df["business_address"].fillna("").map(extract_pin)
pin_to_s1 = set(s1_df.loc[s1_df["pin"] != "", "pin"])

# 2. PIN-blocked pool scan over S2+S3 (vectorized extract per chunk)
t0 = time.time()
pool_parts = []
for name in ["train_source2.tsv", "train_source3.tsv"]:
    for ch in pd.read_csv(DATA_ROOT / "train" / name, sep="\t", chunksize=200000):
        pins = ch["business_address"].fillna("").str.extract(PIN_RE, expand=False)
        hit = ch[pins.isin(pin_to_s1)]
        if len(hit):
            pool_parts.append(hit)
pool_df = pd.concat(pool_parts, ignore_index=True).drop_duplicates("entity_id") if pool_parts else s1_df.iloc[0:0].copy()
pool_df["norm_name"] = pool_df["business_name"].fillna("").map(normalize_text)
pool_df["norm_addr"] = pool_df["business_address"].fillna("").map(normalize_text)
pool_df["pin"] = pool_df["business_address"].fillna("").map(extract_pin)
print(f"pool rows sharing a PIN: {len(pool_df)} in {time.time()-t0:.0f}s")

# 3. candidates per S1 (cap 40 = sanity bound) + untrained score -> matches
pool_by_pin: dict = {}
for r in pool_df.itertuples():
    pool_by_pin.setdefault(r.pin, []).append(r.entity_id)
pool_idx = {r.entity_id: r for r in pool_df.itertuples()}
s1_idx = {r.entity_id: r for r in s1_df.itertuples()}
candidates = {s: list(pool_by_pin.get(s1_idx[s].pin, []))[:40] for s in sample_s1}
print(f"mean K={float(np.mean([len(v) for v in candidates.values()])):.1f}")


def pair_score(a, b) -> float:
    ratio = fuzz.WRatio(a.norm_name, b.norm_name) / 100.0
    ta, tb = set(a.norm_name.split()), set(b.norm_name.split())
    jac = len(ta & tb) / max(1, len(ta | tb))
    pin = 1.0 if (a.pin and a.pin == b.pin) else 0.0
    return 0.5 * ratio + 0.3 * jac + 0.2 * pin


TAU_SKELETON = 0.5
matches = {}
for s in sample_s1:
    a = s1_idx[s]
    scored = sorted(((pool_idx[c], pair_score(a, pool_idx[c])) for c in candidates[s]),
                    key=lambda t: -t[1])
    matches[s] = [r.entity_id for r, sc in scored if sc >= TAU_SKELETON]
print(f"skeleton macro-F0.5 on sample: {macro_f05({s: val_matches[s] for s in sample_s1}, matches):.4f}")
# checkpoint skeleton pairs for fine-tuning below
pd.to_pickle({"candidates": candidates, "sample_s1": sample_s1}, CKPT_DIR / "skeleton.pkl")
print("checkpointed", CKPT_DIR / "skeleton.pkl")

In [ ]:
# write TSVs (tab-separated, utf-8) + validate: coverage, dupes, prefixes, matching ⊆ candidates
def write_id_list(path: Path, rows: dict, col: str):
    df = pd.DataFrame({"source1_entity_id": list(rows.keys()),
                       col: [",".join(rows[k]) for k in rows]})
    df.to_csv(path, sep="\t", index=False, encoding="utf-8")


write_id_list(OUT_DIR / "matching_results.tsv", matches, "matched_entity_ids")
write_id_list(OUT_DIR / "candidate_pairs.tsv", candidates, "candidate_entity_ids")

issues = []
m = pd.read_csv(OUT_DIR / "matching_results.tsv", sep="\t", keep_default_na=False)
c = pd.read_csv(OUT_DIR / "candidate_pairs.tsv", sep="\t", keep_default_na=False)
req = set(sample_s1)
if set(m["source1_entity_id"]) != req or len(m) != len(req):
    issues.append("matching S1 coverage mismatch")
if set(c["source1_entity_id"]) != req or len(c) != len(req):
    issues.append("candidate S1 coverage mismatch")
valid = set(pool_df["entity_id"].astype(str))
cmap = {}
for _, r in c.iterrows():
    ids = [x for x in str(r["candidate_entity_ids"]).split(",") if x]
    if len(ids) != len(set(ids)):
        issues.append(f"dupes in candidates for {r['source1_entity_id']}")
    if any(not x.startswith(("S2-", "S3-")) or x not in valid for x in ids):
        issues.append(f"bad candidate ids for {r['source1_entity_id']}")
    cmap[r["source1_entity_id"]] = set(ids)
for _, r in m.iterrows():
    ids = [x for x in str(r["matched_entity_ids"]).split(",") if x]
    if len(ids) != len(set(ids)):
        issues.append(f"dupes in matches for {r['source1_entity_id']}")
    if not set(ids) <= cmap.get(r["source1_entity_id"], set()):
        issues.append(f"match not in candidates for {r['source1_entity_id']}")
print("VALIDATION:", "PASS" if not issues else f"FAIL {issues[:5]}")

## Phase 2 — GPU ANN blocking at A100 scale (primary index, per plan)
Sentence-transformer embeddings encoded in fp16 batches on GPU, cosine via FAISS `IndexFlatIP` (GPU-resident if `faiss-gpu` present, else CPU index over GPU-encoded vectors — the encode is the expensive part and it runs on the A100 either way). Recall denominator = matches present in pool.

In [ ]:
import faiss
from sentence_transformers import SentenceTransformer

demo_pool = pool_df.head(POOL_DOCS).reset_index(drop=True)
pool_texts = (demo_pool["norm_name"] + " [SEP] " + demo_pool["norm_addr"]).tolist()
q_texts = [(s1_idx[s].norm_name + " [SEP] " + s1_idx[s].norm_addr) for s in sample_s1]
q_ids = [str(x) for x in demo_pool["entity_id"]]
pool_set = set(q_ids)

enc = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device="cuda")  # MIT license
t0 = time.time()
X = enc.encode(pool_texts, batch_size=ENC_BATCH, normalize_embeddings=True,
               show_progress_bar=True).astype("float32")
Q = enc.encode(q_texts, batch_size=ENC_BATCH, normalize_embeddings=True,
               show_progress_bar=True).astype("float32")
index = faiss.IndexFlatIP(X.shape[1])
index.add(X)
D, I = index.search(Q, TOP_K)
print(f"GPU-ANN: pool={len(demo_pool)} dim={X.shape[1]} topK={TOP_K} in {time.time()-t0:.0f}s")

recalls, covered = [], 0
for i, s in enumerate(sample_s1):
    denom = [m for m in val_matches[s] if m in pool_set]
    if not denom:
        continue
    covered += 1
    hits = [q_ids[j] for j in I[i]]
    recalls.append(len(set(hits) & set(denom)) / len(denom))
print(f"pool coverage of sample GT: {covered}/{len(sample_s1)}")
if recalls:
    print(f"recall@{TOP_K} (in-pool denom) mean={np.mean(recalls):.3f}")
print("NOTE: skeleton pool is PIN-biased — index smoke test, not a recall claim.")

## Phase 3 (GPU) — Fine-tune the bi-encoder on skeleton pairs
Positives = GT matches present in candidates; negatives = other candidates for the same S1 (hard negatives for free). `MultipleNegativesRankingLoss`, fp16, checkpoints to Drive. The fine-tuned model is what the local notebook consumes via OpenVINO.

In [ ]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

FT_N = 50000  # S1 queries used for tuning (raise once the smoke run passes)
ft_s1 = sample_s1[:FT_N]
examples = []
for s in ft_s1:
    a = s1_idx[s].norm_name + " [SEP] " + s1_idx[s].norm_addr
    pos = [c for c in candidates[s] if c in set(val_matches[s])]
    neg = [c for c in candidates[s] if c not in set(val_matches[s])][:3]
    for p in pos:
        r = pool_idx.get(p)
        if r is None:
            continue
        pt = r.norm_name + " [SEP] " + r.norm_addr
        nt = [(pool_idx[n].norm_name + " [SEP] " + pool_idx[n].norm_addr) for n in neg if n in pool_idx]
        examples.append(InputExample(texts=[a, pt] + nt))
print(f"tuning pairs: {len(examples)}")
assert len(examples) > 1000, "too few positives — check blocking recall before tuning"

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
loader = DataLoader(examples, batch_size=64, shuffle=True)
loss = losses.MultipleNegativesRankingLoss(model)
model.fit([(loader, loss)], epochs=FT_EPOCHS, warmup_steps=500,
          output_path=str(CKPT_DIR / "minilm-er"), show_progress_bar=True)
print("saved", CKPT_DIR / "minilm-er")

## Phase 3 (GPU) — Cross-encoder rerank demo on sample top-10
Rescores the skeleton's top candidates per S1; threshold on rerank scores is tuned for macro-F0.5 in Phase 4. Small sample here to stay quick — scale to full candidates after.

In [ ]:
from sentence_transformers import CrossEncoder

RERANK_N = 2000
ce = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2", max_length=256, device="cuda")
pairs, keys = [], []
for s in sample_s1[:RERANK_N]:
    a = s1_idx[s].norm_name + " [SEP] " + s1_idx[s].norm_addr
    for c in candidates[s][:TOP_K]:
        if c in pool_idx:
            r = pool_idx[c]
            pairs.append((a, r.norm_name + " [SEP] " + r.norm_addr))
            keys.append((s, c))
scores = ce.predict(pairs, batch_size=ENC_BATCH, show_progress_bar=True)
order = np.argsort(-scores)
print(f"reranked {len(pairs)} pairs; top-5 scores: {sorted(scores, reverse=True)[:5].round(3)}")
pd.to_pickle({"keys": keys, "scores": scores}, CKPT_DIR / "rerank_sample.pkl")
print("checkpointed", CKPT_DIR / "rerank_sample.pkl")

## Export — OpenVINO IR for the local notebook (runs on Linux Colab, consumed on Windows/Arc)
Exports the fine-tuned encoder to OpenVINO IR on Drive. Copy `models/minilm-ov` to the PC; the local notebook's Phase-3 cell compiles it on the Arc 140T.

In [ ]:
FT_MODEL = CKPT_DIR / "minilm-er"
if not FT_MODEL.exists():
    print("no fine-tuned model yet — run the tuning cell first.")
else:
    from optimum.intel import OVModelForFeatureExtraction
    ov_m = OVModelForFeatureExtraction.from_pretrained(str(FT_MODEL), export=True)
    ov_m.save_pretrained(str(OV_DIR))
    print("exported OpenVINO IR to", OV_DIR, sorted(p.name for p in OV_DIR.iterdir()))

## Phase 2 scale-up — full data (flip RUN_FULL with Drive checkpointing on)
Recipe: per-country shard → GPU-encoded embeddings in chunks → FAISS index (FlatIP per shard, IVF-PQ if RAM pinches) → 50k-chunk queries → PIN/token backfill + rank-merge + adaptive cap → streaming candidate writes to Drive → recall gate (≥95% ceiling → minimize mean K per §1.6). Session can die anytime: every shard's candidates flush to Drive before the next starts.

In [ ]:
assert RUN_FULL, (
    "Full-data blocking needs RUN_FULL=True + Drive mounted. Recipe: (1) per-country shard, "
    "(2) GPU-encode chunks -> FAISS, (3) 50k-chunk queries w/ per-shard flush to Drive, "
    "(4) backfill + rank-merge + adaptive cap, (5) recall/K gate on val.")
# full-scale code lands here (Phase 2 of ENTITY_RESOLUTION_PLAN.md)

## Phase 4 / 5 — Roadmap
- **Phase 4:** tune global `tau` (rerank or fused scores) on val macro-F0.5 (expect ~0.6–0.8); `max_score < tau` → empty list. Per-country `tau` only on clear drift.
- **Phase 5:** chunked test inference on GPU → both TSVs to Drive → `validate_submission.py --check-ids` must PASS → submission zip.

In [ ]:
import json
print(json.dumps({"mode": MODE, "gpu": GPU, "sample_S1": len(sample_s1),
                   "pool_rows": len(pool_df), "drive": str(BASE),
                   "outputs": sorted(str(p) for p in OUT_DIR.glob("*.tsv"))}, indent=2))